In [14]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [15]:
df = pd.read_csv("../data/cleaned_realistic_ocean_climate.csv")

In [16]:
df.shape

(500, 6)

In [17]:
df.head()

,Latitude,Longitude,SST (°C),pH Level,Bleaching Severity,Marine Heatwave
0,20.0248,38.4931,29.47,8.107,0,False
1,-18.2988,147.7782,29.65,8.004,2,False
2,14.9768,-75.0233,28.86,7.947,2,False
3,-18.3152,147.6486,28.97,7.995,1,False
4,-0.8805,-90.9769,28.60,7.977,0,False


In [18]:
X = df.drop(columns=["Bleaching Severity"])
y = df["Bleaching Severity"]

In [19]:
X.columns.tolist()
y.value_counts()

Bleaching Severity
0    282
1    130
2     88
Name: count, dtype: int64

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test) 

In [22]:
joblib.dump(scaler, "../models/scaler_nn_ROC.pkl")

['../models/scaler_nn_ROC.pkl']

In [23]:
num_features = X_train_scaled.shape[1]
num_classes  = 3

In [24]:
model = keras.Sequential([
    layers.Input(shape=(num_features,)),
    
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(16, activation="relu"),
    
    layers.Dense(num_classes, activation="softmax")  # output 3 classes
])

In [25]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,427 (13.39 KB)

 Trainable params: 3,235 (12.64 KB)

 Non-trainable params: 192 (768.00 B)

In [26]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # ใช้เมื่อ y เป็น integer (0,1,2)
    metrics=["accuracy"]
)

In [27]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5
    )
]

In [28]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.2719 - loss: 1.3691 - val_accuracy: 0.4125 - val_loss: 1.0903 - learning_rate: 0.0010
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3625 - loss: 1.2592 - val_accuracy: 0.5500 - val_loss: 1.0695 - learning_rate: 0.0010
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3656 - loss: 1.0942 - val_accuracy: 0.5750 - val_loss: 1.0569 - learning_rate: 0.0010
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4062 - loss: 1.0940 - val_accuracy: 0.5875 - val_loss: 1.0458 - learning_rate: 0.0010
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4281 - loss: 1.0816 - val_accuracy: 0.6000 - val_loss: 1.0352 - learning_rate: 0.0010
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4750 - loss: 1.0097 - val_accuracy: 0.5875 - val_loss: 1.0245 - learning_rate: 0.0010
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4938 - loss: 1.0843 - val_ac

In [29]:
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {accuracy:.4f}")
print(f"✅ Test Loss: {loss:.4f}")
y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))


✅ Test Accuracy: 0.5600
✅ Test Loss: 0.9826
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step

Classification Report:
              precision    recall  f1-score   support

         Low       0.56      1.00      0.72        56
      Medium       0.00      0.00      0.00        26
        High       0.00      0.00      0.00        18

    accuracy                           0.56       100
   macro avg       0.19      0.33      0.24       100
weighted avg       0.31      0.56      0.40       100



C:\Users\Jirayut Pimmuen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Jirayut Pimmuen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Jirayut Pimmuen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-

In [ ]:
model.save("../models/neural_network_ROC.h5")